# 05 — Generation

In [1]:
import os
if not os.path.exists('MiniGPT'):
    !git clone https://github.com/userKk1/MiniGPT.git
%cd MiniGPT

Cloning into 'MiniGPT'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 149 (delta 72), reused 66 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 255.24 KiB | 15.95 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/MiniGPT


In [2]:
import sys, json
sys.path.append('.')

import torch
from config import cfg, DATA_PROCESSED_DIR, CHECKPOINTS_DIR
from src.model import GPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 1. Load vocab and the best checkpoint

`ckpt_best.pt` (lowest val loss) is what we want here, not necessarily
the final training step's weights.

In [3]:
!python -m src.data

README.md: 100% 1.30k/1.30k [00:00<00:00, 5.12MB/s]
Resolving data files: 100% 54/54 [00:00<00:00, 24512.17it/s]
Collected 10294 files, 50.00 MB
Wrote /content/MiniGPT/data/raw/corpus_raw.txt (52.4 MB)


In [4]:
!python -m src.tokenizer

Corpus length: 52,404,086 characters
Training BPE tokenizer...
[00:00:00] Tokenize words                 ██████████████████ 245824   /   245824
[00:00:00] Count pairs                    ██████████████████ 245824   /   245824
[00:00:01] Compute merges                 ██████████████████ 898      /      898
BPE vocab size: 1,000

Encoding entire corpus...
Total BPE tokens: 22,581,723
train: 20,323,550 tokens
val:   2,258,173 tokens

Done.
Saved train data: /content/MiniGPT/data/processed/train.bin
Saved val data:   /content/MiniGPT/data/processed/val.bin
Saved tokenizer:  /content/MiniGPT/data/processed/tokenizer.json


In [5]:
!python -m src.train

device: cuda
vocab_size: 1000  train tokens: 20,323,550  val tokens: 2,258,173
Parameters: 5,312,000
step     0 | train 6.9336 | val 6.9328 | lr 1.50e-06 | 9s
step   250 | train 4.3671 | val 4.4052 | lr 3.00e-04 | 49s
step   500 | train 3.8264 | val 3.8571 | lr 2.99e-04 | 91s
step   750 | train 3.5289 | val 3.5856 | lr 2.98e-04 | 138s
step  1000 | train 3.3501 | val 3.3933 | lr 2.96e-04 | 182s
step  1250 | train 3.1670 | val 3.2067 | lr 2.92e-04 | 227s
step  1500 | train 3.0130 | val 3.0638 | lr 2.88e-04 | 272s
step  1750 | train 2.8839 | val 2.9596 | lr 2.84e-04 | 318s
step  2000 | train 2.7367 | val 2.8000 | lr 2.78e-04 | 362s
step  2250 | train 2.5861 | val 2.6683 | lr 2.72e-04 | 407s
step  2500 | train 2.4489 | val 2.5028 | lr 2.65e-04 | 452s
step  2750 | train 2.3365 | val 2.4226 | lr 2.57e-04 | 497s
step  3000 | train 2.2622 | val 2.3468 | lr 2.49e-04 | 542s
step  3250 | train 2.2140 | val 2.2849 | lr 2.40e-04 | 587s
step  3500 | train 2.1780 | val 2.2385 | lr 2.31e-04 | 632s
ste

In [6]:
from tokenizers import Tokenizer

In [7]:
tokenizer = Tokenizer.from_file(
    str(DATA_PROCESSED_DIR / "tokenizer.json")
)
vocab_size = tokenizer.get_vocab_size()

print(f"Vocabulary size: {vocab_size}")

model = GPT(vocab_size).to(device)
ckpt = torch.load(CHECKPOINTS_DIR / 'ckpt_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Loaded checkpoint from step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f}")

Vocabulary size: 1000
Loaded checkpoint from step 9500, val_loss=1.9038


## 2. Sampling strategies, briefly

- **Greedy** (temperature -> 0): always pick the single most likely next
  token. Deterministic, often repetitive/loops on itself.
- **Temperature**: scale logits before softmax. Low temperature (e.g. 0.3)
  sharpens the distribution (safer, more repetitive); high (e.g. 1.2) flattens
  it (more variety, more mistakes).
- **Top-k**: only sample from the k most likely tokens at each step — cuts
  off the long low-probability tail that produces nonsense.
- **Top-p (nucleus)**: like top-k, but the cutoff is dynamic — keep the
  smallest set of tokens whose cumulative probability exceeds p. Adapts to
  how confident the model is at each step, unlike top-k's fixed count.

In [8]:
def generate_from_prompt(
    prompt,
    max_new_tokens=200,
    **kwargs
):
    encoded = tokenizer.encode(prompt)

    idx = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
        device=device
    )

    # Generate new token IDs
    out = model.generate(
        idx,
        max_new_tokens=max_new_tokens,
        **kwargs
    )


    # Convert token IDs → Python code
    return tokenizer.decode(out[0].tolist())

## 3. Compare strategies on the same prompt

Using a real code prefix as the prompt — this is the actual 'code
completion' use case, closer to what you'd demo than generating from an
empty context.

In [9]:
prompt = 'def factorial(n):\n    '

print('=== Greedy (temperature=0.01 , repetition_penalty=1.3) ===')
print(generate_from_prompt(prompt, temperature=0.01,repetition_penalty=1.3))
print()

print('=== Temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8))
print()

print('=== Top-k=40, temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8, top_k=40))
print()

print('=== Top-p=0.9, temperature=0.8 ===')
print(generate_from_prompt(prompt, temperature=0.8, top_p=0.9))

=== Greedy (temperature=0.01 , repetition_penalty=1.3) ===
def factorial(n):
     return n

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
This module is a generated by the terms of the GNU Lesser Generator.

The Software Foundation, either version 3 of the License, or (at your option) any later version.

The Software is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

=== Temperature=0.8 ===
def factorial(n):
     # stuffsets here
         multi_params = None
        
#     upper binary
#     if upper binary in sorted(multi_params):
#     upper binary in want to run the binary form in c.get_binary(c.get_binary(n))
#     else:
#         a = ""
#     r = ""
#     if r > 0:
#         a = ""
#         a = ""
#         a = ""
#         r.add("")
#         r.add("")
#         r.add("", "")
#         r.add(

## 4. Try a few more realistic completion prompts

Pick prompts similar to what a real code-completion demo would show —
partial function signatures, class definitions, import blocks.

In [10]:
prompts = [
    'class Config:\n    def __init__(self):\n        ',
    'import numpy as np\n\ndef ',
    'def test_',
]

for p in prompts:
    print('PROMPT:', repr(p))
    print(generate_from_prompt(p, max_new_tokens=300, temperature=0.8, top_p=0.9, repetition_penalty=1.3))
    print('-' * 60)

PROMPT: 'class Config:\n    def __init__(self):\n        '
class Config:
    def __init__(self):
         self.config = config[0]
    
        for locals, cmd in os.walk(cmd)
        if not cmd['Method']:
            logger.info("Server data in connecting")
            failure = True
        else:
            done_str = "" + str(locals()) + " download")
            shutil.rmtree(failures=True) 
          	else:
                raise ValueError('Additioned by the Node')
    
    def getOpenThreadChanged(self, cmd):
        # For couldn't handle it about optional strings in this work to remove authentication of device here
        return False
    
    @staticmethod
    def stopBodyWindow(self, cmd):
        """Generate anything text."""
        pass
        
    
    def generateColumnWidget(self):
        """Define manager that source changes"""
        name = "generateColumnActionInfo"
        name = self._object._getPathLayout()
       
-----------------------------------------------

## 5. Save a few samples for your results/

These go straight into `results/samples.md` — concrete evidence of what
the model produces, at a given checkpoint/step.

In [11]:
from config import RESULTS_DIR

lines = [f"# Sample generations (checkpoint step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f})", '']
for p in prompts:
    completion = generate_from_prompt(p, max_new_tokens=150, temperature=0.8, top_p=0.9)
    lines.append(f'## Prompt: `{p!r}`')
    lines.append('```python')
    lines.append(completion)
    lines.append('```')
    lines.append('')

(RESULTS_DIR / 'samples.md').write_text('\n'.join(lines))
print(f"Saved to {RESULTS_DIR / 'samples.md'}")

Saved to /content/MiniGPT/results/samples.md
